# Lesson 5 · Electric fields and potential

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/05_electric_fields.ipynb)

**For:** A-level Physics (for example AQA 3.7.1 and 3.7.3, OCR A 6.2, Edexcel Topic 7) and IB Physics (D.2 electric and magnetic fields). Check your own specification for the exact wording.

**You will be able to:**

1. use Coulomb's law $F = kQ_1Q_2/r^2$, and compare electric and gravitational forces
2. use $E = F/q$ and $E = kQ/r^2$, and add fields from several charges as vectors
3. use $V = kQ/r$, $E = -dV/dr$ and $W = q\Delta V$
4. sketch field lines and equipotentials, and explain why they are perpendicular
5. use $E = V/d$ for a uniform field, and explain edge effects

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Exercises are at the end, with worked solutions below them.

In [ ]:
# Setup: installs the qplay package if it is missing (for example on Google Colab, about 20 s).
import importlib.util, subprocess, sys
if importlib.util.find_spec("qplay") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/StativeSh/quantum-playground"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import em

k = em.K_E                       # Coulomb constant 1/(4 pi epsilon_0), about 8.99e9 N m^2 C^-2
e = 1.602e-19                    # C
m_e, m_p = 9.109e-31, 1.673e-27  # kg
G = 6.674e-11
print(f"k = {k:.4e} N m^2 C^-2")

## 1. Coulomb's law

Two point charges $Q_1$ and $Q_2$ a distance $r$ apart exert equal and opposite forces on each other:

$$F = \frac{1}{4\pi\varepsilon_0}\frac{Q_1Q_2}{r^2} = k\frac{Q_1Q_2}{r^2}.$$

Like charges repel, unlike charges attract. It is an inverse-square law, exactly like gravity, but vastly stronger. Compare the two forces between a pair of protons:

In [ ]:
r = 1e-15                                     # m, about the size of a nucleus   <- Try it: any r
F_electric = k * e**2 / r**2
F_gravity = G * m_p**2 / r**2
print(f"electric force  {F_electric:.3e} N")
print(f"gravity         {F_gravity:.3e} N")
print(f"ratio           {F_electric / F_gravity:.3e}  (the same at every distance: both fall as 1/r^2)")

Gravity is about $10^{36}$ times weaker, so inside atoms and nuclei it can be ignored. (Something else, the strong force, must hold nuclei together against this repulsion.)

## 2. Electric field strength

The **electric field strength** at a point is the force per unit positive charge placed there, $E = F/q$, in N C$^{-1}$. Around a point charge $Q$,

$$E = \frac{kQ}{r^2}.$$

Fields from several charges add as **vectors**. Below, the package adds the fields of two charges at every point of a grid and draws field lines (the direction a small positive charge would be pushed).

In [ ]:
xs = np.linspace(-2, 2, 241)
X, Y = np.meshgrid(xs, xs)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, charges, title in [(axes[0], [1, -1], "Dipole: +Q and -Q"), (axes[1], [1, 1], "Two equal positive charges")]:
    positions = [(-0.7, 0), (0.7, 0)]
    Ex, Ey = em.point_charge_field(charges, positions, X, Y)
    V = em.point_charge_potential(charges, positions, X, Y)
    strength = np.log10(np.clip(np.hypot(Ex, Ey), 0.05, 20))     # clip: the field is infinite at the charges
    ax.streamplot(X, Y, Ex, Ey, color=strength, cmap="viridis", density=1.4, linewidth=0.8)
    ax.contour(X, Y, V, levels=np.linspace(-3, 3, 13), colors="C3", linewidths=0.7, linestyles="solid")
    for q, (px, py) in zip(charges, positions):
        ax.plot(px, py, "o", ms=12, color="C3" if q > 0 else "C0")
        ax.text(px, py, "+" if q > 0 else "−", color="w", ha="center", va="center", fontsize=12, weight="bold")
    ax.set(aspect="equal", xlim=(-2, 2), ylim=(-2, 2), title=title)
plt.suptitle("Field lines (coloured by strength) and equipotentials (red)");

Things to notice:

- Field lines start on positive charges and end on negative ones; they never cross.
- Lines are close together where the field is strong.
- Between the two positive charges there is a point where the fields cancel ($E = 0$): a **neutral point**.
- The red **equipotential** lines always cross field lines at right angles.

## 3. Electric potential

The **electric potential** $V$ at a point is the work done per unit charge bringing a small positive charge from infinity to that point. Around a point charge,

$$V = \frac{kQ}{r},$$

a scalar (no direction), so potentials from several charges simply add as numbers. The field is the **potential gradient**, pointing downhill:

$$E = -\frac{dV}{dr}.$$

The work done moving a charge $q$ between two points is $W = q\,\Delta V$. Below we check $E = -dV/dr$ numerically for a single charge.

In [ ]:
Q = 5e-9                                      # C
r = np.linspace(0.05, 0.5, 400)
V = k * Q / r
E_from_gradient = -np.gradient(V, r)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
ax1.plot(r, V); ax1.set(xlabel="r / m", ylabel="V / V", title=r"Potential $V = kQ/r$")
ax2.plot(r, E_from_gradient, label=r"$-dV/dr$ (numerical)")
ax2.plot(r[::25], k * Q / r[::25] ** 2, "o", ms=4, label=r"$kQ/r^2$")
ax2.set(xlabel="r / m", ylabel=r"E / N C$^{-1}$", title="Field = minus the potential gradient"); ax2.legend()
print(f"largest relative difference: {np.max(np.abs(E_from_gradient[1:-1] / (k * Q / r[1:-1]**2) - 1)):.1e}")

**Exam tip:** on a graph of $V$ against $r$, the field strength is minus the **gradient**. On a graph of $E$ against $r$, the potential difference between two points is the **area** under the curve.

## 4. Uniform fields between parallel plates

Between two large parallel plates with potential difference $V$ and separation $d$, the field is uniform:

$$E = \frac{V}{d}.$$

The package can solve Laplace's equation $\nabla^2 V = 0$ for the potential in a box. Here the top plate is at 100 V, the bottom plate at 0 V, and the side walls are earthed (0 V).

In [ ]:
d, width = 0.02, 0.08                          # plate separation and width in m
nx, ny = 161, 41
V = em.solve_laplace(nx, ny, top=100.0, bottom=0.0, left=0.0, right=0.0)
x = np.linspace(0, width, nx); y = np.linspace(0, d, ny)
dVdy, dVdx = np.gradient(V, y, x)
Ex, Ey = -dVdx, -dVdy

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [2, 1]})
cs = ax1.contourf(x * 100, y * 100, V, levels=21, cmap="coolwarm")
ax1.streamplot(x * 100, y * 100, Ex, Ey, color="k", density=0.8, linewidth=0.6)
fig.colorbar(cs, ax=ax1, label="V / V"); ax1.set(xlabel="x / cm", ylabel="y / cm", aspect="equal", title="Equipotentials and field lines")
middle = nx // 2
ax2.plot(y * 100, V[:, middle], label="along the middle")
ax2.plot(y * 100, V[:, 8], label="near a side wall")
ax2.set(xlabel="y / cm", ylabel="V / V", title="Potential across the gap"); ax2.legend()
print(f"|E| in the middle = {abs(dVdy[ny // 2, middle]):.1f} V/m (pointing from the 100 V plate to the 0 V plate);   V/d = {100 / d:.1f} V/m")

In the middle the potential rises in a straight line, so the field is uniform and equal to $V/d$. Near the earthed side walls the field is weaker and bends: these are **edge effects**, which is why "large" plates (wide compared with their separation) are assumed.

A charge $q$ in a uniform field feels a constant force $F = qE = qV/d$, just as a mass feels a constant force near the Earth's surface. Lesson 6 follows charged particles through such fields.

---

## Exercises

Take $k = 8.99\times10^{9}\ \mathrm{N\,m^2\,C^{-2}}$ and $e = 1.60\times10^{-19}$ C. Give answers to 3 significant figures.

**Q1.** Find the size of the force between charges of $+3.0\ \mu$C and $-5.0\ \mu$C that are 0.12 m apart. Is it attractive or repulsive?

**Q2.** For a point charge of $+2.0$ nC, find (a) the field strength and (b) the potential at a distance of 0.50 m.

**Q3.** A charge of $+4.0\ \mu$C is at $x = 0$ and a charge of $+1.0\ \mu$C is at $x = 0.30$ m. Where between them is the electric field zero?

**Q4.** Two parallel plates are 2.0 cm apart with a potential difference of 500 V between them. Find (a) the field strength between them, (b) the force on an electron, (c) the electron's acceleration ($m_e = 9.11\times10^{-31}$ kg).

**Q5.** How much work is needed to move a $+2.0\ \mu$C charge from 0.40 m to 0.10 m away from a fixed $+5.0\ \mu$C charge?

**Q6 (simulation).** Charges of $+2.0$ nC and $-2.0$ nC are at $(-0.03, 0)$ m and $(+0.03, 0)$ m. Use `em.point_charge_field(..., k=k)` to find the field strength at $(0, 0.04)$ m.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_r6 = np.hypot(0.03, 0.04)
_answers = {
    "Q1": k * 3.0e-6 * 5.0e-6 / 0.12**2,
    "Q2a": k * 2.0e-9 / 0.50**2, "Q2b": k * 2.0e-9 / 0.50,
    "Q3": 0.30 * np.sqrt(4.0) / (np.sqrt(4.0) + np.sqrt(1.0)),      # 4/x^2 = 1/(0.3 - x)^2
    "Q4a": 500 / 0.020, "Q4b": e * 500 / 0.020, "Q4c": e * 500 / 0.020 / m_e,
    "Q5": 2.0e-6 * k * 5.0e-6 * (1 / 0.10 - 1 / 0.40),
    "Q6": 2 * k * 2.0e-9 / _r6**2 * (0.03 / _r6),                    # y-components cancel, x-components add
}

# Replace None with your answers, in SI units (Q1 as a positive number, Q3 as the x coordinate in m).
mine = {"Q1": None, "Q2a": None, "Q2b": None, "Q3": None, "Q4a": None, "Q4b": None, "Q4c": None, "Q5": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** $F = \dfrac{kQ_1Q_2}{r^2} = \dfrac{8.99\times10^9 \times 3.0\times10^{-6} \times 5.0\times10^{-6}}{0.12^2} = 9.36\ \mathrm{N}$, attractive because the charges have opposite signs.

**Q2.** (a) $E = kQ/r^2 = 8.99\times10^9 \times 2.0\times10^{-9}/0.25 = 71.9\ \mathrm{N\,C^{-1}}$ &nbsp; (b) $V = kQ/r = 8.99\times10^9 \times 2.0\times10^{-9}/0.50 = 36.0\ \mathrm{V}$

**Q3.** At distance $x$ from the 4.0 µC charge the two fields are equal and opposite: $\dfrac{k(4.0\,\mu\mathrm C)}{x^2} = \dfrac{k(1.0\,\mu\mathrm C)}{(0.30 - x)^2}$. Taking square roots, $2(0.30 - x) = x$, so $x = 0.20$ m. It is closer to the smaller charge, as it must be.

**Q4.** (a) $E = V/d = 500/0.020 = 2.50\times10^{4}\ \mathrm{V\,m^{-1}}$ &nbsp; (b) $F = eE = 1.60\times10^{-19} \times 2.5\times10^4 = 4.00\times10^{-15}\ \mathrm{N}$ &nbsp; (c) $a = F/m = 4.40\times10^{15}\ \mathrm{m\,s^{-2}}$. Gravity ($9.81\ \mathrm{m\,s^{-2}}$) is utterly negligible by comparison.

**Q5.** $W = q\,\Delta V = qkQ\left(\dfrac{1}{0.10} - \dfrac{1}{0.40}\right) = 2.0\times10^{-6} \times 8.99\times10^9 \times 5.0\times10^{-6} \times 7.5 = 0.674\ \mathrm{J}$

**Q6.** Each charge is $r = \sqrt{0.03^2 + 0.04^2} = 0.05$ m from the point, so each field has size $kq/r^2 = 7190\ \mathrm{N\,C^{-1}}$. The $y$-components cancel and the $x$-components add: $E = 2 \times 7190 \times \dfrac{0.03}{0.05} = 8.63\times10^{3}\ \mathrm{N\,C^{-1}}$, pointing from the positive charge's side towards the negative one. The cell below checks this with the package.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}")
Ex6, Ey6 = em.point_charge_field([2.0e-9, -2.0e-9], [(-0.03, 0), (0.03, 0)], np.array([[0.0]]), np.array([[0.04]]), k=k)
print(f"package: Ex = {Ex6[0, 0]:.4g} N/C, Ey = {Ey6[0, 0]:.2g} N/C")

**Next:** in section 2, change the charges to `[2, -1]`. Where is the neutral point now, and why is it outside the pair of charges?